## Model 1
ResNet-50 & MLP (Multi Layer Perceptron) -> folosit in cazul in care se extrag manual feature-uri din metadate

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class MultimodalBreastCancerModel1(nn.Module):
    def __init__(self, num_tabular_features=3, num_classes=3, dropout_rate=0.3, classifier="linear", projection_dim=256):

        super(MultimodalBreastCancerModel1, self).__init__()

        resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.image_extractor = nn.Sequential(*list(resnet.children())[:-1])
        image_embed_size = resnet.fc.in_features # 2048

        # 66.666% conteaza procesarea imaginii si 33.333% datele textuale
        img_proj_dim = 512
        tab_proj_dim = 256

        self.tabular_extractor = nn.Sequential(
            nn.Linear(num_tabular_features, 128),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(128, tab_proj_dim),
            nn.ReLU()
        )

        self.image_projector = nn.Sequential(
            nn.Linear(image_embed_size, img_proj_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )

        # 512 (imagine) + 256 (tabel) = 768 dimensiuni combinate
        combined_size = img_proj_dim + tab_proj_dim

        self.dropout = nn.Dropout(p=dropout_rate)

        # Clasificatorul final
        self.classifier = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, images, tabular_data):
        img_features = self.image_extractor(images)
        img_features = torch.flatten(img_features, 1)

        tab_features = self.tabular_extractor(tabular_data)

        # se aplica proiectia pe imagine
        img_proj = self.image_projector(img_features)

        fused_features = torch.cat((img_proj, tab_features), dim=1)

        x = self.dropout(fused_features)
        logits = self.classifier(x)

        return logits

## Model 2
ResNet-50 & ClinicalBERT -> folosit in cazul in care se analizeaza O PARTE din fisa medicala a pacientului

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
from transformers import AutoModel

class MultimodalBreastCancerModel2(nn.Module):
    def __init__(self, num_classes=3, dropout_rate=0.3, classifier="mlp"):
        super(MultimodalBreastCancerModel2, self).__init__()

        resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.image_extractor = nn.Sequential(*list(resnet.children())[:-1])
        image_embed_size = resnet.fc.in_features # Corectat: variabilă locală (2048)

        self.text_extractor = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        text_embed_size = self.text_extractor.config.hidden_size # 768

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.text_extractor.to(device)

        # 50% conteaza imaginea si 50% fisa medicala
        img_proj_dim = 512
        text_proj_dim = 512

        self.image_projector = nn.Sequential(
            nn.Linear(image_embed_size, img_proj_dim),
            nn.BatchNorm1d(img_proj_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )

        self.text_projector = nn.Sequential(
            nn.Linear(text_embed_size, text_proj_dim),
            nn.BatchNorm1d(text_proj_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )

        # img_proj_dim + text_proj_dim (1024)
        combined_size = img_proj_dim + text_proj_dim

        self.dropout = nn.Dropout(p=dropout_rate)

        # Clasificator
        if classifier == "linear":
            self.classifier = nn.Linear(combined_size, num_classes)
        elif classifier == "mlp":
            self.classifier = nn.Sequential(
                nn.Linear(combined_size, 512),
                nn.ReLU(),
                nn.Dropout(p=dropout_rate),
                nn.Linear(512, num_classes)
            )

    def forward(self, images, input_ids, attention_mask):
        img_features = self.image_extractor(images)
        img_features = torch.flatten(img_features, 1)

        text_outputs = self.text_extractor(input_ids=input_ids, attention_mask=attention_mask)
        #text_features = text_outputs.pooler_output

        # MEAN POOLING (inlocuieste pooler_output)
        token_embeddings = text_outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        text_features = sum_embeddings / sum_mask

        # Se aplica proiectiile
        img_proj = self.image_projector(img_features)
        text_proj = self.text_projector(text_features)

        fused_features = torch.cat((img_proj, text_proj), dim=1)

        x = self.dropout(fused_features)
        logits = self.classifier(x)

        return logits

## Clasa de formare a dataset-urilor + preprocesare
Pentru utilizarea coloanelor din metadata.csv

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image

class BreastCancerDataset(Dataset):
    def __init__(self, df_merged, png_dir, transform=None):
        self.data = df_merged.copy()
        self.png_dir = png_dir
        self.transform = transform

        # Feature-urile tabelare extrase din metadata.csv
        base_features = ['Age', 'Breast', 'Breast composition', 'Mass_Present', 'Calcifications_Present', 'Calcifications_Type']
        one_hot_features = [col for col in self.data.columns if any(prefix in col for prefix in ['Mass_Shape_', 'Mass_Margin_', 'Mass_Localization_', 'Calcifications_Distribution_'])]
        self.selected_features = base_features + one_hot_features

        # Preprocesare tabele (convertim Yes/No in 1/0, curatam NaN)
        self.data['Age'] = pd.to_numeric(self.data['Age'], errors='coerce').fillna(-1)

        if 'Breast' in self.data.columns:
            self.data['Breast'] = self.data['Breast'].map({'right': 1, 'left': 0}).fillna(-1)

        if 'Breast composition' in self.data.columns:
            self.data['Breast composition'] = self.data['Breast composition'].map({'C': 0, 'D': 1}).fillna(-1)

        if 'Mass_Present' in self.data.columns:
            self.data['Mass_Present'] = self.data['Mass_Present'].map({'no': 0, 'yes': 1}).fillna(-1)

        if 'Mass_Shape' in self.data.columns:
            self.data['Mass_Shape'] = self.data['Mass_Shape'].map({'oval/round': 0, 'lobulated': 1, 'irregular':2}).fillna(-1)

        if 'Mass_Margin' in self.data.columns:
            self.data['Mass_Margin'] = self.data['Mass_Margin'].map({'circumscribed': 0, 'obscured': 1,'microlobulated':2,'indistinct':3,'spiculated':4}).fillna(-1)

        if 'Mass_Localization' in self.data.columns:
            self.data['Mass_Localization'] = self.data['Mass_Localization'].map({'quadrant-based': 0, 'retromammary': 1,'dispersed':2,'entire breast':3}).fillna(-1)

        if 'Calcifications_Present' in self.data.columns:
            self.data['Calcifications_Present'] = self.data['Calcifications_Present'].map({'no': 0, 'yes': 1}).fillna(-1)

        if 'Calcifications_Type' in self.data.columns:
            self.data['Calcifications_Type'] = self.data['Calcifications_Type'].map({'micro': 0, 'macro': 1}).fillna(-1)

        if 'Calcifications_Distribution' in self.data.columns:
            self.data['Calcifications_Distribution'] = self.data['Calcifications_Distribution'].map({'Diffuse': 0, 'Regional': 1,'Grouped':2,'Linear':3,'Segmental':4}).fillna(-1)


        self.data['Classification'] = self.data['Classification'].map({'healthy': 0, 'benign': 1, 'malignant': 2})

        # Se vor elimina pacientii pentru care nu avem un diagnostic concret
        self.data = self.data.dropna(subset=['Classification']).reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Alegerea imaginilor
        # Deoarece folderul este plat, punem direct numele fisierului PNG extras din image_types.csv
        img_name = row['Image_name']
        img_path = os.path.join(self.png_dir, img_name)

        image_pil = Image.open(img_path).convert("RGB")
        image_np = np.array(image_pil)

        if self.transform:
            image = self.transform(image=image_np)['image']
        else:
            image = torch.tensor(image_np).permute(2, 0, 1).float() / 255.0

        # Datele tabelare extrase din metadata.csv
        features = row[self.selected_features].values.astype(float)
        tabular_data = torch.tensor(features, dtype=torch.float32)

        label = int(row['Classification'])

        return {
            'image': image,
            'tabular_data': tabular_data,
            'label': torch.tensor(label, dtype=torch.long)
        }

## Clasa de formare a dataset-urilor + preprocesare
Pentru utilizarea fiselor medicale

In [ ]:
import re
import pandas as pd

def elimina_giveaways(text):
    if pd.isna(text):
        return text

    # se elimina orice variatie de BI-RADS
    text = re.sub(r'(?i)bi-?rads\s*\d+[a-c]?', '', text)

    # se inlocuiesc cuvintele directe legate de diagnostic cu cuvantul neutru "lesion"
    text = re.sub(r'(?i)\b(tumor|tumour|malignant|malignancy|benign|cancer|carcinoma)\b', 'lesion', text)

    # se elimina fraze specifice care indica direct un stadiu avansat
    text = re.sub(r'(?i)advanced loco-regional stage', '', text)

    # se elimina toti termenii procedurali (biopsie, ace, teste de laborator, markeri)
    termeni_proceduri = r'(?i)\b(biopsy|puncture|tissue (fragments?|samples?)|pathology( laboratory)?|cytology|histology|histopathological|metallic clip|marker|xylocaine|anesthesia|14 g(\s?needle|\s?biopsy gun)?|automatic gun|core needle|fna|tru-cut|vacuum-assisted|hemostasis|immediate local complications|ultrasound guidance|image-guided)\b'
    text = re.sub(termeni_proceduri, '', text)

    # se curata spatiile duble sau punctuatia ramasa izolata
    text = re.sub(r'\s+', ' ', text).strip()

    return text


from transformers import AutoTokenizer

class BreastCancerTextDataset(Dataset):
    def __init__(self, df, png_dir, tokenizer, max_length=128, transform=None):
        self.data = df.copy()
        self.png_dir = png_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.data['Classification'] = self.data['Classification'].map({'healthy': 0, 'benign': 1, 'malignant': 2})
        # se elimina cazurile nediagnosticate
        self.data = self.data.dropna(subset=['Classification']).reset_index(drop=True)

        self.data['Cleaned_Report'] = self.data['Cleaned_Report'].fillna("No report available")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        img_name = row['Image_name']
        img_path = os.path.join(self.png_dir, img_name)

        image_pil = Image.open(img_path).convert("RGB")
        image_np = np.array(image_pil)

        if self.transform:
            image = self.transform(image=image_np)['image']
        else:
            image = torch.tensor(image_np).permute(2, 0, 1).float() / 255.0

        text = str(row['Cleaned_Report'])

        # tokenizarea Textului pentru ClinicalBERT
        # se transforma cuvintele in indexuri si se adauga padding ca toate textele sa aiba aceeasi lungime
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        label = int(row['Classification'])

        # .squeeze(0) pentru eliminarea dimensiunii extra pusa de tokenizer
        return {
            'image': image,
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

## Impartirea datelor in train, val si test

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

def get_dataloaders(df_merged, png_dir, transform_train=None, transform_test=None, batch_size=16,dataset='data'):
    pacienti_unici = df_merged['Patient_id'].unique()

    # 75% Train, 25% Temp (Val + Test)
    pacienti_train, pacienti_temp = train_test_split(pacienti_unici, test_size=0.25, random_state=42)

    # 15% Validare, 10% Test
    pacienti_val, pacienti_test = train_test_split(pacienti_temp, test_size=0.4, random_state=42)

    # Se obtin dataset-urile pentru training, validare, respectiv testare
    df_train = df_merged[df_merged['Patient_id'].isin(pacienti_train)].reset_index(drop=True)
    df_val = df_merged[df_merged['Patient_id'].isin(pacienti_val)].reset_index(drop=True)
    df_test = df_merged[df_merged['Patient_id'].isin(pacienti_test)].reset_index(drop=True)

    print(f"Distributie imagini: Train={len(df_train)}, Val={len(df_val)}, Test={len(df_test)}")

    if dataset == 1:
        train_dataset = BreastCancerDataset(df_train, png_dir, transform=transform_train)
        val_dataset = BreastCancerDataset(df_val, png_dir, transform=transform_test)
        test_dataset = BreastCancerDataset(df_test, png_dir, transform=transform_test)
    elif dataset == 2:
        print("Se descarca Tokenizer-ul pentru ClinicalBERT...")
        tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

        train_dataset = BreastCancerTextDataset(
            df=df_train,
            png_dir=png_dir,
            tokenizer=tokenizer,
            max_length=128,
            transform=transform_train
        )

        val_dataset = BreastCancerTextDataset(
            df=df_val,
            png_dir=png_dir,
            tokenizer=tokenizer,
            max_length=128,
            transform=transform_test
        )

        test_dataset = BreastCancerTextDataset(
            df=df_test,
            png_dir=png_dir,
            tokenizer=tokenizer,
            max_length=128,
            transform=transform_test
        )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,num_workers=2,pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

## Train + preprocesare pentru integrarea cu modelul

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
import pandas as pd
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Trebuie redimensionate mamografiile pentru a se potrivi cu ResNet-50 (224x224)
'''
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # augmentare (nu se va intoarce imaginea pe vertical -> schimba semnificatia medicala)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # normalizare standard ResNet
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
'''

transform_train = A.Compose([
    A.Resize(512, 512),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0), # Scoate în evidență tumorile în sânii denși
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

transform_test = A.Compose([
    A.Resize(512, 512),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])


def run_training_phase(train_loader, val_loader,model_type=1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n---> Pornim antrenamentul pe: {device} <---")

    if model_type == 1:
        nr_caracteristici_noi = len(train_loader.dataset.selected_features)
        model = MultimodalBreastCancerModel1(num_tabular_features=nr_caracteristici_noi, num_classes=3, classifier="mlp").to(device)
    elif model_type == 2:
        model = MultimodalBreastCancerModel2(num_classes=3, dropout_rate=0.4,classifier="mlp").to(device)

    # [Healthy, Benign, Malignant]
    #class_weights = torch.tensor([1.0, 1.2, 1.5], dtype=torch.float32).to(device)

    # Extragem etichetele direct din dataset pentru a calcula ponderile corecte
    all_train_labels = [train_loader.dataset[i]['label'].item() for i in range(len(train_loader.dataset))]
    class_weights_vals = compute_class_weight(class_weight='balanced', classes=np.array([0, 1, 2]), y=all_train_labels)
    class_weights = torch.tensor(class_weights_vals, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    criterion = nn.CrossEntropyLoss()

    # Rata de invatare -> mai mica pentru ResNet (deoarece este deja antrenat partial)
    #                  -> mai mare pentru MLP-ul nou

    if model_type == 1:
        optimizer = optim.Adam([
            {'params': model.image_extractor.parameters(), 'lr': 1e-4},
            {'params': model.image_projector.parameters(), 'lr': 1e-4},
            {'params': model.tabular_extractor.parameters(), 'lr': 1e-4},
            {'params': model.classifier.parameters(), 'lr': 1e-4}
        ])
    elif model_type == 2:
        optimizer = optim.Adam([
        {'params': model.image_extractor.parameters(), 'lr': 1e-5},
        {'params': model.image_projector.parameters(), 'lr': 5e-5},
        {'params': model.text_extractor.parameters(), 'lr': 1e-5},
        {'params': model.text_projector.parameters(), 'lr': 5e-5},
        {'params': model.classifier.parameters(), 'lr': 1e-4}
    ],weight_decay = 1e-2)

    # mode='min' -> vrem ca val_loss sa scada
    # factor=0.5 -> injumatateste LR-ul
    # patience=2 -> așteapta 2 epoci fara imbunatatiri inainte sa reduca LR-ul
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    best_val_loss = float('inf')
    num_epochs = 5

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        train_bar = tqdm(train_loader, desc=f"Epoca {epoch+1}/{num_epochs} [TRAIN]")

        # Incepe bucla de antrenare
        for batch in train_bar:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)

            if model_type == 1:
                tabular_data = batch['tabular_data'].to(device)

                # Forward pass
                logits = model(images, tabular_data)

            elif model_type == 2:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                # Forward pass
                logits = model(images, input_ids, attention_mask)


            optimizer.zero_grad()

            # Calcularea erorii
            loss = criterion(logits, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_bar.set_postfix(loss=loss.item())

        # Bucla de validare
        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(device)
                labels = batch['label'].to(device)

                if model_type == 1:
                    tabular_data = batch['tabular_data'].to(device)
                    logits = model(images, tabular_data)
                elif model_type == 2:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    logits = model(images, input_ids, attention_mask)

                loss = criterion(logits, labels)
                val_loss += loss.item()

                predictions = torch.argmax(logits, dim=1)
                all_preds.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = accuracy_score(all_labels, all_preds)

        print(f"Epoca {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

        scheduler.step(avg_val_loss)

        # Se printeaza noul learning rate
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Rata de învățare curentă este: {current_lr}")

        # Se salveaza modelul doar daca s-a imbunatatit
        if avg_val_loss < best_val_loss:
            print(">>> Nou model de top gasit! Salvăm 'best_multimodal_model.pth'...")
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_multimodal_model.pth")

## Testare

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def run_testing_phase(test_loader, model_type = 1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n--- INIȚIEM TESTAREA FINALA (TEST SET) ---")

    if model_type == 1:
        nr_caracteristici_noi = len(train_loader.dataset.selected_features)
        model = MultimodalBreastCancerModel1(num_tabular_features=nr_caracteristici_noi, num_classes=3, classifier="mlp").to(device)
    elif model_type == 2:
        model = MultimodalBreastCancerModel2(num_classes=3, dropout_rate=0.4,classifier="mlp").to(device)

    try:
        model.load_state_dict(torch.load("best_multimodal_model.pth"))
        print("Am incarcat cu succes 'best_multimodal_model.pth'!")
    except FileNotFoundError:
        print("Eroare: Nu am gasit fisierul. Trebuie sa rulezi run_training_phase() mai intai!")
        return


    model.eval()
    test_preds, test_labels = [], []

    with torch.no_grad(): # se opreste calculul de gradienti -> nu mai invata

        for batch in tqdm(test_loader, desc="Testare"):
            images = batch['image'].to(device)
            labels = batch['label'].to(device)

            if model_type == 1:
                tabular_data = batch['tabular_data'].to(device)
                # Se face predictia
                logits = model(images, tabular_data)
            elif model_type == 2:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                # Se face predictia
                logits = model(images, input_ids, attention_mask)

            predictions = torch.argmax(logits, dim=1)

            # Se salveaza rezultatele
            test_preds.extend(predictions.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    # Calcularea metricilor finale (acuratete, precizie, rapel si scorul F1)
    acc = accuracy_score(test_labels, test_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='weighted')

    print(f"\nREZULTATE FINALE TESTARE:")
    print(f"Acuratete: {acc:.4f}")
    print(f"Precizie: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Scor F1:   {f1:.4f}")

    print("\n--- RAPORT DETALIAT PE CLASE ---")
    print(classification_report(test_labels, test_preds, target_names=['Healthy', 'Benign', 'Malignant']))

## Main

In [ ]:
import os
import pandas as pd

if __name__ == "__main__":

    cale_metadata = "/kaggle/input/datasets/bajcsiadel96/romanian-dense-breast-mammography-collection/metadata.csv"
    png_dir = "/kaggle/input/datasets/bajcsiadel96/romanian-dense-breast-mammography-collection/pngs"

    print("Se genereaza setul de date unificat (df_merged)...")

    df_metadata = pd.read_csv(cale_metadata)

    toate_fisierele = os.listdir(png_dir)
    date_imagini = []

    for fisier in toate_fisierele:
        # Citirea imaginilor
        if fisier.endswith('_MAM.png'):
            parti = fisier.split('_')

            if len(parti) >= 4:
                cod_extras = f"{parti[0]}_{parti[1]}_{parti[2]}_{parti[3]}"
                patient_id = f"{parti[0]}_{parti[1]}"
                date_imagini.append({
                    'Image_name': fisier,
                    'Code': cod_extras,
                    'Patient_id': patient_id
                })

    df_imagini = pd.DataFrame(date_imagini)
    df_merged = pd.merge(df_imagini, df_metadata, on='Code', how='inner')

    coloane_categorice = ['Mass_Shape', 'Mass_Margin', 'Mass_Localization', 'Calcifications_Distribution']
    df_merged = pd.get_dummies(df_merged, columns=coloane_categorice, dtype=float)

    print("Se curata rapoartele medicale de cuvinte indici ...")
    df_merged['Cleaned_Report'] = df_merged['Ultrasound report_English'].apply(elimina_giveaways)

    print("Se impart datele...")
    train_loader, val_loader, test_loader = get_dataloaders(
        df_merged=df_merged,
        png_dir=png_dir,
        transform_train=transform_train,
        transform_test=transform_test,
        batch_size=16,
        dataset = 1
    )

    print("Se incepe antrenamentul!")
    run_training_phase(train_loader, val_loader,model_type = 1)

    print("Se incepe testarea finala!")
    run_testing_phase(test_loader, model_type = 1)